# KV Cache in LLMs

## The Problem It Solves

When a transformer generates text, every new token needs to attend to all previous tokens via the attention mechanism. Without caching, generating each new token requires recomputing the Key and Value matrices for every previous token from scratch — an enormous waste since those tokens haven't changed.

---

## How Attention Works (Brief)

In transformer attention, every token produces three vectors:

```
Query (Q)  — "what am I looking for?"
Key   (K)  — "what do I contain?"
Value (V)  — "what information do I hold?"

Attention(Q, K, V) = softmax(QKᵀ / √d) × V
```

When generating token #100, the model needs K and V from all 99 previous tokens. Without cache it recomputes them every single time.

---

## What KV Cache Does

Store the Key and Value matrices from previous tokens so they don't need to be recomputed. Only the new token's Q, K, V need to be computed on each step.

```
Without KV Cache — generating token N:
  Compute K, V for tokens 1, 2, 3 ... N-1, N  ← wasteful
  
With KV Cache — generating token N:
  Load K, V for tokens 1, 2, 3 ... N-1  ← from cache
  Compute K, V only for token N          ← new computation
```

---

## Visual Illustration

```
Prompt: "The capital of France is"  → tokens [1, 2, 3, 4, 5]

Step 1 — Prefill phase (process entire prompt at once)
┌────────────────────────────────────┐
│  Compute K,V for all prompt tokens │
│  Store in KV Cache                 │
│  Generate first output token       │
└────────────────────────────────────┘
KV Cache: [K1,V1] [K2,V2] [K3,V3] [K4,V4] [K5,V5]

Step 2 — Decode phase (generate "Paris")
┌────────────────────────────────────┐
│  Load K,V from cache (tokens 1-5)  │
│  Compute K,V only for new token    │
│  Attend to all tokens → "Paris"    │
└────────────────────────────────────┘
KV Cache: [K1,V1] ... [K5,V5] [K6,V6]  ← append new token

Step 3 — Generate next token
KV Cache grows by one entry each step
```

---

## Code Illustration

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model     = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

prompt  = "The capital of France is"
inputs  = tokenizer(prompt, return_tensors="pt")

# WITHOUT KV cache — recomputes everything each step
output_no_cache = model.generate(
    **inputs,
    max_new_tokens=10,
    use_cache=False   # recompute K,V for all tokens every step
)

# WITH KV cache — stores and reuses K,V
output_with_cache = model.generate(
    **inputs,
    max_new_tokens=10,
    use_cache=True    # default, reuse cached K,V
)

# Both produce same output, cache version is much faster
```

### What's Inside the Cache

```python
# Manually inspect KV cache
with torch.no_grad():
    outputs = model(
        **inputs,
        use_cache=True,
        return_dict=True
    )

past_key_values = outputs.past_key_values

print(f"Layers in cache : {len(past_key_values)}")
print(f"K shape per layer: {past_key_values[0][0].shape}")
# torch.Size([1, num_heads, seq_len, head_dim])
# (batch, heads, tokens, dimension)

# On next forward pass, feed the cache back in
next_token_input = torch.tensor([[next_token_id]])
outputs2 = model(
    input_ids=next_token_input,
    past_key_values=past_key_values,  # pass cached K,V
    use_cache=True
)
```

---

## Memory Cost of KV Cache

KV cache trades compute for memory. As sequence length grows, cache size grows linearly.

```
Cache size = 2              # K and V
           × num_layers
           × num_heads
           × seq_length
           × head_dim
           × bytes_per_element

# Example: LLaMA 3 8B, FP16, 4096 token context
cache_size = 2 × 32 × 32 × 4096 × 128 × 2 bytes
           = 2 GB

# At 32k context (long document):
cache_size = 2 × 32 × 32 × 32768 × 128 × 2 bytes
           = 16 GB  ← just for the cache!
```

This is why long context inference is expensive and why KV cache management is a major challenge in production LLM serving.

---

## KV Cache Optimizations

### Multi-Query Attention (MQA)
Instead of separate K, V per head, share one K, V across all heads. Reduces cache size by num_heads times.

```python
# Standard Multi-Head Attention
# Q: [batch, num_heads, seq, head_dim]
# K: [batch, num_heads, seq, head_dim]  ← one per head
# V: [batch, num_heads, seq, head_dim]

# Multi-Query Attention (MQA)
# Q: [batch, num_heads, seq, head_dim]
# K: [batch, 1,         seq, head_dim]  ← shared!
# V: [batch, 1,         seq, head_dim]  ← shared!
```

### Grouped Query Attention (GQA)
Middle ground — groups of heads share K, V. Used by LLaMA 3, Mistral, Gemma.

```python
# GQA — 8 query heads share 2 KV heads (4:1 ratio)
# Q: [batch, 8, seq, head_dim]
# K: [batch, 2, seq, head_dim]  ← 4x smaller cache
# V: [batch, 2, seq, head_dim]
```

### Sliding Window Attention
Only cache the last N tokens instead of all tokens. Used in Mistral — reduces memory while retaining local context.

### PagedAttention (vLLM)
Manages KV cache like virtual memory pages, eliminating fragmentation and enabling much better GPU utilization for serving multiple requests.

```python
# vLLM uses PagedAttention internally
from vllm import LLM, SamplingParams

llm = LLM(model="meta-llama/Llama-3.2-8B")
# Automatically uses PagedAttention for efficient KV cache management

outputs = llm.generate(
    ["Tell me about KV cache"],
    SamplingParams(temperature=0.7, max_tokens=200)
)
```

---

## KV Cache in Production

```
Without KV Cache:
  Token generation time ∝ O(n²)  ← gets slower as sequence grows

With KV Cache:
  Token generation time ∝ O(n)   ← constant per new token

Real world speedup: 5-30x faster generation
```

### Prefix Caching
If many requests share the same system prompt, cache that prefix once and reuse across all requests — huge savings in production.

```python
# vLLM automatic prefix caching
llm = LLM(
    model="llama3",
    enable_prefix_caching=True  # cache shared system prompts
)

# All requests with same system prompt reuse cached K,V
requests = [
    "You are a helpful assistant. [User query 1]",
    "You are a helpful assistant. [User query 2]",  # prefix cache hit
    "You are a helpful assistant. [User query 3]",  # prefix cache hit
]
```

---

## Summary

| Aspect | Detail |
|---|---|
| What it stores | Key and Value matrices from attention layers |
| What it saves | Recomputing K,V for previous tokens |
| Speedup | 5-30x faster token generation |
| Cost | Memory grows linearly with sequence length |
| Key variants | MQA, GQA, Sliding Window, PagedAttention |
| Used by | Every production LLM inference system |

KV cache is one of those foundational optimizations that makes LLM serving economically viable — without it, generating long responses would be so slow and compute-intensive that most real-world applications simply wouldn't work.